## Pull in UDFs

In [1]:
%run nb_udfs

StatementMeta(, 3b4d8538-6f90-4df2-9961-5a96c7a6d443, 13, Finished, Available, Finished, True)

## Run Workspaces

In [2]:
workspace = 'Fabric of Middle-Earth' #have to escape the & symbol using %26%
lakehouse = 'lh_the_hoard'
ws_table = 'dimWorkspaces'
workspace_users_table = 'dimWorkspaceUsers'
current_user = mssparkutils.env.getUserName()

StatementMeta(, 3b4d8538-6f90-4df2-9961-5a96c7a6d443, 14, Finished, Available, Finished, False)

In [10]:
#get a list of all workspaces and load to a table in the lakehouse
response = fab_client.get(f"/v1/admin/workspaces")
df_workspaces = pd.json_normalize(response.json()['workspaces'])
#df_workspaces
df_workspaces = spark.createDataFrame(df_workspaces).filter(col("type") != "Personal")

#df_workspaces.show(1000,truncate=False)

StatementMeta(, 3b4d8538-6f90-4df2-9961-5a96c7a6d443, 93, Finished, Available, Finished, False)

+------------------------------------+--------------------------------------------------------+------+--------------+------------------------------------+------------------------------------+
|id                                  |name                                                    |state |type          |capacityId                          |domainId                            |
+------------------------------------+--------------------------------------------------------+------+--------------+------------------------------------+------------------------------------+
|31e584e2-91d6-4c95-8fda-37c763c6c59b|DS.Adworks_L1                                           |Active|Workspace     |7BD00511-FB4D-42FD-911B-596C053A4DBB|NULL                                |
|46777af4-fa87-459e-ba70-88730f3a2b00|QA.Sales_L1                                             |Active|Workspace     |77CC05EE-A02F-40A9-8244-F44A47BF468D|NULL                                |
|ae858ee6-1ef6-4c99-8056-76d0c9c09d67|DS

In [29]:
#creates a slowly changing dimension so we can keep an eye on any deleted workspaces
workspacePath = udf_GetFilePath(workspace, lakehouse, ws_table)
print(workspacePath)
naturalKeyColumnList = ['id']
primaryKeyColumnName = "tableId"
returnVal = udf_UpsertDimension(df_workspaces,2,workspacePath,naturalKeyColumnList,primaryKeyColumnName,True)
print(returnVal)

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 61, Finished, Available, Finished)

abfss://253a31af-7442-450f-b93a-680a4831c265@onelake.dfs.fabric.microsoft.com/7ea68a16-ce87-4757-aaaf-48a22e04038c/Tables/dimWorkspaces
✅ Upsert complete
{'startTime': '2025-09-09 19:40:08.110526', 'stopTime': '2025-09-09 19:40:17.395529', 'details': '0 records updated, 0 records inserted from 34 staging rows to abfss://253a31af-7442-450f-b93a-680a4831c265@onelake.dfs.fabric.microsoft.com/7ea68a16-ce87-4757-aaaf-48a22e04038c/Tables/dimWorkspaces'}


## Get the Workspace Users

In [30]:
#creates a list of workspaces we want users for
df_np_workspaces = df_workspaces \
    .filter(df_workspaces["type"] == "Workspace") 

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 62, Finished, Available, Finished)

In [31]:
def process_workspace_users(row):
    workspace_id, workspace_name = row['id'], row['name']
    request = f"/v1.0/myorg/groups/{workspace_id}/users"
    #admin rest api only allows for 200 calls an hour. Using non admin so we can go through all workspaces much faster
    #request = f"/v1.0/myorg/admin/groups/{workspace_id}/users"
    result = {'workspace_users': None, 'workspace_id': workspace_id, 'workspace_name': workspace_name, 'error': None}
    try:
        response = _base_api(request=request, method="get")
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))
            print(f"⛔️ 429 on {workspace_name}. Sleeping {retry_after}s, then retrying once...")
            time.sleep(retry_after)
            response = _base_api(request=request, method="get")
    except Exception as e:
        result['error'] = f"Exception: {e}"
        return result

    # check the status before parsing JSON; empty or HTML bodies caused "Expecting value: line 1 column 1 (char 0)"
    if response.status_code != 200:
        result['error'] = f"HTTP {response.status_code}: {response.text[:200]!r}"
        return result
    try:
        users = response.json().get('value')
    except ValueError:
        result['error'] = f"Non-JSON body (HTTP {response.status_code}): {response.text[:200]!r}"
        return result
    if users is None:
        result['error'] = "No 'value' key in response"
        return result
    if not users:
        return result  # workspace with no users; createDataFrame can't infer a schema from an empty list

    df_workspace_users = spark.createDataFrame(pd.json_normalize(users))
    df_workspace_users = df_workspace_users.withColumn("workspaceId", lit(workspace_id))
    df_workspace_users = df_workspace_users.withColumn("compositeKey", concat_ws("", col("workspaceId"), col("emailAddress"), col("identifier")))
    result['workspace_users'] = df_workspace_users
    return result

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 63, Finished, Available, Finished)

In [32]:
rows_workspaces = df_np_workspaces.collect()
results_workspace_users, failures = [], []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures_wu = {executor.submit(process_workspace_users, row): row for row in rows_workspaces}
    for future in as_completed(futures_wu):
        row = futures_wu[future]
        try:
            result = future.result()
        except Exception as e:
            failures.append((row['name'], row['id'], f"Unhandled: {e}"))
            continue
        if result['error']:
            failures.append((result['workspace_name'], result['workspace_id'], result['error']))
        elif result['workspace_users'] is not None:
            results_workspace_users.append(result['workspace_users'])

print(f"✅ Looping complete: {len(results_workspace_users)} workspaces with users, {len(failures)} failed")
for name, wid, err in failures:
    print(f"❌ {name} ({wid}): {err}")

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 64, Finished, Available, Finished)

✅ Looping complete!


In [33]:
#union the results together
if results_workspace_users:
    df_workspace_users = union_batches(results_workspace_users,batch_size=50)
    print("✅ workspace users dataframe ready")
else:
    print("⚠️ No workspace users found.")



StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 65, Finished, Available, Finished)

✅ workspace users dataframe ready


In [34]:
df_workspace_users.show(100)

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 66, Finished, Available, Finished)

+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+
|groupUserAccessRight|         displayName|          identifier|principalType|        emailAddress|         workspaceId|        compositeKey|
+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+
|               Admin|   Power BI Security|4b42412e-2041-431...|        Group|                null|37237e02-1d16-400...|37237e02-1d16-400...|
|               Admin|     Kristyna Ferris|kristyna@dataonwh...|         User|kristyna@dataonwh...|37237e02-1d16-400...|37237e02-1d16-400...|
|               Admin|       Kontoor Admin|cf631fa8-2e2f-42e...|        Group|                null|31e584e2-91d6-4c9...|31e584e2-91d6-4c9...|
|              Viewer|     Adworks_Cont_L1|a5c0e2f0-bfca-4b9...|        Group|                null|31e584e2-91d6-4c9...|31e584e2-91d6-4c9...|
|     

In [35]:
#creates a slowly changing dimension so we can keep an eye on any removed workspace users
workspaceUserPath = udf_GetFilePath(workspace, lakehouse, workspace_users_table)
print(workspaceUserPath)
naturalKeyColumnList = ['compositeKey']
primaryKeyColumnName = "tableId"
#skip soft deletes when any workspace failed so its users aren't marked deleted
returnVal = udf_UpsertDimension(df_workspace_users,2,workspaceUserPath,naturalKeyColumnList,primaryKeyColumnName,flagSoftDeletes=(len(failures) == 0))
print(returnVal)

StatementMeta(, 6812e845-df71-4728-85b1-b1f2240796ba, 67, Finished, Cancelled, Cancelled)

abfss://253a31af-7442-450f-b93a-680a4831c265@onelake.dfs.fabric.microsoft.com/7ea68a16-ce87-4757-aaaf-48a22e04038c/Tables/dimWorkspaceUsers


Py4JJavaError: An error occurred while calling o8676.javaToPython.
: org.apache.spark.SparkException: Job 117 cancelled part of cancelled job group 67
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2871)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:2746)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$4(DAGScheduler.scala:1187)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.java:23)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1186)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3029)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3009)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:2998)
	at org.apache.spark.util.EventLoop$$